Drug Response Chatbot

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd
df= pd.read_csv("GDSC_DATASET.csv")

Data Preprocessing

In [ ]:
df['TCGA_DESC'] = df['TCGA_DESC'].fillna("Unknown")
df['Cancer Type (matching TCGA label)'] = df['Cancer Type (matching TCGA label)'].fillna("Unknown")

In [ ]:
high_value_cols = [
    'TARGET',
    'Microsatellite instability Status (MSI)',
    'GDSC Tissue descriptor 1',
    'GDSC Tissue descriptor 2'
]

for col in high_value_cols:
    df[col + "_missing"] = df[col].isnull().astype(int)
    df[col] = df[col].fillna("Unknown")

In [ ]:

medium_cols = ['Screen Medium', 'Growth Properties']

for col in medium_cols:
    df[col] = df[col].fillna("Unknown")

In [ ]:
bio_cols = ['CNA', 'Gene Expression', 'Methylation']

bio_cols = ['CNA', 'Gene Expression', 'Methylation']

for col in bio_cols:
    df[col] = df[col].fillna("Unknown")

In [ ]:
df = df.drop_duplicates()

In [ ]:
df = df[['DRUG_NAME', 'DRUG_ID', 'LN_IC50', 'AUC', 'Z_SCORE','TARGET','TARGET_PATHWAY','CELL_LINE_NAME','Cancer Type (matching TCGA label)','TCGA_DESC']]

df = df.dropna()

Drug-Level Aggregation

In [ ]:
drug_summary = df.groupby('DRUG_NAME').agg({
    'TARGET': lambda x: ', '.join(set(x)),
    'TARGET_PATHWAY': lambda x: ', '.join(set(x)),
    'CELL_LINE_NAME': lambda x: ', '.join(set(x)),
    'Cancer Type (matching TCGA label)': lambda x: ', '.join(set(x)),
    'LN_IC50': ['mean', 'min', 'max'],
    'AUC': ['mean']
}).reset_index()

In [ ]:
drug_summary.columns = [
    'DRUG_NAME',
    'TARGET',
    'PATHWAY',
    'CELL_LINES',
    'CANCER_TYPES',
    'IC50_MEAN',
    'IC50_MIN',
    'IC50_MAX',
    'AUC_MEAN'
]

Text Generation

In [ ]:
def create_text(row):
    return f"""
    Drug: {row['DRUG_NAME']}.
    Target: {row['TARGET']}.
    Pathway: {row['PATHWAY']}.
    Cancer types: {row['CANCER_TYPES']}.
    Cell lines: {row['CELL_LINES']}.
    IC50 ranges from {row['IC50_MIN']} to {row['IC50_MAX']} with mean {row['IC50_MEAN']}.
    """

drug_summary['text'] = drug_summary.apply(create_text, axis=1)

Embedding Model (Semantic Understanding)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(drug_summary['text'].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/9 [00:00<?, ?it/s]

FAISS Vector Database

In [ ]:
!pip install faiss-cpu
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

Retrieval Function (Smart Matching)

In [ ]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    results = drug_summary.iloc[indices[0]]

    # ✅ PRIORITIZE exact match
    exact_match = results[
        results['DRUG_NAME'].str.lower() == query.lower()
    ]

    if not exact_match.empty:
        return exact_match

    return results

BioGPT Generation

In [ ]:
!pip install sacremoses
from transformers import pipeline
generator = pipeline(
    "text-generation",
    model="microsoft/BioGPT",
    device=-1
)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie biogpt.embed_tokens.weight to output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Prompt Engineering

In [ ]:
def generate_answer(query, retrieved_data):

    # ✅ Extract drug name properly
    drug_name = query.lower().replace("tell me about ", "").strip()

    # ✅ FILTER HERE
    retrieved_filtered = retrieved_data[
        retrieved_data['DRUG_NAME'].str.lower() == drug_name
    ]

    # ⚠️ fallback if nothing found
    if retrieved_filtered.empty:
        retrieved_filtered = retrieved_data.head(1)

    # ✅ Limit size
    retrieved_filtered = retrieved_filtered.head(1)


    context = retrieved_filtered.iloc[0]['text']
    context = context[:1000]

    prompt = f"""
You are a biomedical expert.

Answer ONLY about the drug: {drug_name}

Do NOT include other drugs.

Provide:
- Target
- Pathway
- Cancer types affected
- IC50 range

Context:
{context}

Question: {query}

Answer:
"""

    result = generator(
        prompt,
        max_new_tokens=150,
        temperature=0.7,
        do_sample=True,
        truncation=True
    )

    return result[0]['generated_text']

RAG (Retrieval + Generation)

In [ ]:
query = " Camptothecin"

retrieved = retrieve(query)
answer = generate_answer(query, retrieved)

print(answer)

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a biomedical expert.

Answer ONLY about the drug: camptothecin

Do NOT include other drugs.

Provide:
- Target
- Pathway
- Cancer types affected
- IC50 range

Context:

    Drug: Camptothecin.
    Target: TOP1.
    Pathway: DNA replication.
    Cancer types: LGG, DLBC, THCA, STAD, CLL, ACC, MM, LCML, SCLC, ALL, LUAD, ESCA, GBM, SKCM, HNSC, MB, NB, PRAD, Unknown, OV, CESC, UCEC, LUSC, KIRC, LIHC, PAAD, LAML, MESO, BRCA, COAD/READ, UNABLE TO CLASSIFY, BLCA.
    Cell lines: CFPAC-1, HO-1-u-1, HCC1500, BICR31, NCI-H1688, RPMI-8402, D-336MG, EW-16, NKM-1, NCI-H716, HPAF-II, RCH-ACV, NCI-H1975, NCI-H1666, RT4, WM793B, U-CH2, HOP-92, OSC-20, HuO-3N1, MS751, HT-1080, COR-L321, COLO-680N, HCC-78, BONNA-12, JiyoyeP-2003, HSC-2, BE-13, NCI-H2030, CAPAN-2, NCI-H2595, HDLM-2, MHH-PREB-1, UM-UC-3, JHOS-4, KYAE-1, OMC-1, CML-T1, SW1783, RKN, JHH-7, BICR78, LB647-SCLC, AN3-CA, HCC-827, EW-12, SK-OV-3, SK-UT-1, SLVL, SW48, NAMALWA, CHSA0108, CGTH-W-1, U-87-MG, SNU-81, Detroit562, ESO26, U-266,

Streamlit App (Frontend)

In [ ]:
!pip install streamlit
import streamlit as st

st.title("💊 Drug Response Chatbot")

query = st.text_input("Enter drug name:")

if query:
    retrieved = retrieve(query)
    answer = generate_answer(query, retrieved)

    st.write("### 🧠 Answer")
    st.write(answer)

    st.write("### 🔍 Retrieved Data")
    st.dataframe(retrieved)

2026-03-28 15:18:18.939 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.940 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.941 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.943 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.945 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.947 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.950 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-28 15:18:18.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar